In [1]:
!pip install -q torch lightgbm shap lime statsmodels scikit-learn pandas numpy scipy matplotlib seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [1]:
import os
import pandas as pd
import requests

BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles"
OUT = "nhanes_csv"
os.makedirs(OUT, exist_ok=True)

# --- pick what you need -------------------------------------------------
# key = file code (goes in the URL), value = what it is
FILES = {
    # demographics
    "DEMO_L":   "Demographics (age, sex, race, weights)",

    # blood panel / CBC
    "CBC_L":    "Complete Blood Count w/ 5-part differential",
    "BIOPRO_L": "Standard Biochemistry Profile",
    "FERTIN_L": "Ferritin",
    "TFR_L":    "Transferrin receptor",
    "FOLATE_L": "RBC folate",
    "HSCRP_L":  "High-sensitivity CRP",
    "VID_L":    "Vitamin D",

    # glycemic
    "GHB_L":    "Glycohemoglobin (HbA1c)",
    "GLU_L":    "Plasma fasting glucose",
    "INS_L":    "Insulin",

    # lipids
    "TCHOL_L":  "Total cholesterol",
    "HDL_L":    "HDL",
    "TRIGLY_L": "LDL & triglycerides",

    # questionnaire -> disease labels
    "DIQ_L":    "Diabetes questionnaire",
    "MCQ_L":    "Medical conditions (heart disease, anemia, etc.)",
    "BPQ_L":    "Blood pressure & cholesterol questionnaire",
}


def fetch(code):
    """Download one .XPT and return it as a DataFrame."""
    url = f"{BASE}/{code}.xpt"
    local = os.path.join(OUT, f"{code}.xpt")

    if not os.path.exists(local):
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        with open(local, "wb") as f:
            f.write(r.content)

    # pandas reads SAS transport natively - no SAS install needed
    df = pd.read_sas(local, format="xport")

    # byte strings -> normal strings
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].str.decode("utf-8", errors="ignore")

    return df


frames = {}
for code, desc in FILES.items():
    try:
        df = fetch(code)
        csv_path = os.path.join(OUT, f"{code}.csv")
        df.to_csv(csv_path, index=False)
        frames[code] = df
        print(f"OK   {code:10s} {df.shape[0]:>6} rows x {df.shape[1]:>3} cols  <- {desc}")
    except Exception as e:
        print(f"SKIP {code:10s} {e}")

# --- merge everything on SEQN (respondent id) ---------------------------
if frames:
    merged = None
    for code, df in frames.items():
        if "SEQN" not in df.columns:
            continue
        # Prefix non-key columns with the file code to avoid collisions during merge.
        # This ensures unique column names when merging different datasets.
        rename_cols = {col: f"{col}_{code}" for col in df.columns if col != 'SEQN'}
        df_to_merge = df.rename(columns=rename_cols)

        merged = df_to_merge if merged is None else merged.merge(df_to_merge, on="SEQN", how="outer")

    merged.to_csv(os.path.join(OUT, "nhanes_2021_2023_merged.csv"), index=False)
    print(f"\nmerged -> {merged.shape[0]} rows x {merged.shape[1]} cols")
    print(f"files written to ./{OUT}/")

OK   DEMO_L      11933 rows x  27 cols  <- Demographics (age, sex, race, weights)
OK   CBC_L        8727 rows x  23 cols  <- Complete Blood Count w/ 5-part differential
OK   BIOPRO_L     7199 rows x  42 cols  <- Standard Biochemistry Profile
OK   FERTIN_L     2564 rows x   4 cols  <- Ferritin
OK   TFR_L        2564 rows x   4 cols  <- Transferrin receptor
OK   FOLATE_L     8727 rows x   4 cols  <- RBC folate
OK   HSCRP_L      8727 rows x   4 cols  <- High-sensitivity CRP
OK   VID_L        8727 rows x  10 cols  <- Vitamin D
OK   GHB_L        7199 rows x   3 cols  <- Glycohemoglobin (HbA1c)
OK   GLU_L        3996 rows x   4 cols  <- Plasma fasting glucose
OK   INS_L        3996 rows x   5 cols  <- Insulin
OK   TCHOL_L      8068 rows x   4 cols  <- Total cholesterol
OK   HDL_L        8068 rows x   4 cols  <- HDL
OK   TRIGLY_L     3996 rows x  10 cols  <- LDL & triglycerides
OK   DIQ_L       11744 rows x   9 cols  <- Diabetes questionnaire
OK   MCQ_L       11744 rows x  35 cols  <- Medical

### Implementation

In [ ]:
"""
================================================================================
 NHANES August 2021 - August 2023 ("_L" cycle)
 GLYCEMIC STATUS CLASSIFICATION: Normal / Prediabetes / Diabetes

 A leakage-aware, explainable benchmark of 10 tabular models,
 with a three-arm sensitivity analysis.

 PIPELINE
   1. Download 18 NHANES components straight from CDC, merge on SEQN
   2. Preprocessing: sentinel repair, SI-duplicate removal, leakage control,
      informative-missingness indicators, fold-internal median imputation
      + quantile normalisation
   3. Target validation against self-reported diagnosis (DIQ010)
   4. Univariate screening: ANOVA F + Kruskal-Wallis, Benjamini-Hochberg FDR
   5. 5x5 repeated stratified cross-validation, 10 models, per arm
   6. ~30 evaluation metrics per fold
   7. Statistics: Shapiro-Wilk, Levene, one-way ANOVA + Tukey HSD,
      Friedman + Nemenyi critical difference
   8. Explainability: SHAP (Tree + Kernel), LIME, NAM shape functions,
      TabNet attention masks, LightGBM gain, SHAP-LIME agreement
   9. Cross-arm comparison

 TARGET (American Diabetes Association HbA1c criteria)
   0 Normal        HbA1c <  5.7 %
   1 Prediabetes   HbA1c 5.7-6.4 %
   2 Diabetes      HbA1c >= 6.5 %

 SENSITIVITY ARMS
   full        every non-leaking feature
   labs_only   questionnaire items removed (no self-reported care status)
   strict      labs_only, minus race and minus glucose-adjacent analytes
               (serum osmolality, fasting insulin)

--------------------------------------------------------------------------------
 SETUP (Google Colab)
--------------------------------------------------------------------------------
   !pip install -q torch lightgbm shap lime statsmodels \
                   scikit-learn pandas numpy scipy matplotlib seaborn

   # nothing to upload: CONFIG['download'] = True fetches from CDC directly.
   # To use your own CSVs instead:
   #     CONFIG['download'] = False
   #     CONFIG['data_dir'] = '/content'

 RUNTIME (per arm)
   n_repeats=1 (5 folds)  : ~25 min CPU  /  ~6 min GPU
   n_repeats=5 (25 folds) : ~2.5 h  CPU  /  ~30 min GPU
   Start with CONFIG['n_repeats'] = 1 and CONFIG['arms'] = ['full'].

 DATA SOURCE
   CDC/NCHS, National Health and Nutrition Examination Survey,
   August 2021 - August 2023.
   https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?Cycle=2021-2023
================================================================================
"""

import os
import re
import glob
import json
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats

warnings.filterwarnings('ignore')



# ========================================================================
# SECTION 1 - DATA: DOWNLOAD, MERGE, CLEAN, TARGET, FEATURE ARMS
# ========================================================================

import glob, os, re
import numpy as np
import pandas as pd

XPT_BASE = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles'
XPT_ZERO = 5.397605e-79          # IBM-hex underflow artifact from XPT->CSV
MISSING_CODES = {7, 9, 77, 99, 777, 999, 7777, 9999}
QUESTIONNAIRE_PREFIX = ('MCQ', 'BPQ', 'DIQ', 'AGQ', 'OSQ', 'SMQ', 'PAQ')

# Every component the pipeline knows how to use. Missing ones are skipped.
NHANES_CODES = [
    'DEMO_L',                                          # demographics
    'BIOPRO_L', 'CBC_L', 'GHB_L', 'GLU_L', 'INS_L',    # core labs
    'TRIGLY_L', 'TCHOL_L', 'HDL_L',                    # lipids
    'HSCRP_L', 'VID_L', 'FERTIN_L', 'TFR_L',           # inflammation / micronutrient
    'BMX_L', 'BPXO_L',                                 # body measures, blood pressure
    'MCQ_L', 'BPQ_L', 'DIQ_L',                         # questionnaires
]
REQUIRED = ['DEMO_L', 'GHB_L']    # age/sex, and HbA1c which IS the target

CANDIDATE_DIRS = ['.', '/content', '/content/nhanes', '/content/data',
                  '/content/drive/MyDrive', '/content/drive/MyDrive/nhanes',
                  '/kaggle/input', '/kaggle/working', '/mnt/user-data/uploads']


# --------------------------------------------------------------------------
# download
# --------------------------------------------------------------------------
def download_nhanes(out_dir='./nhanes_csv', codes=None, overwrite=False):
    """Fetch XPT files straight from CDC and convert to CSV.

    Avoids the upload dance entirely — pandas reads SAS transport natively, so
    no SAS install and no conversion tool is needed.
    """
    codes = codes or NHANES_CODES
    os.makedirs(out_dir, exist_ok=True)
    ok, failed = [], []
    for c in codes:
        dest = os.path.join(out_dir, f'{c}.csv')
        if os.path.exists(dest) and not overwrite:
            ok.append(c)
            print(f'  [cached] {c}')
            continue
        try:
            df = pd.read_sas(f'{XPT_BASE}/{c}.xpt', format='xport')
            for col in df.columns:
                if df[col].dtype == object:
                    df[col] = df[col].str.decode('utf-8', errors='ignore')
            df.to_csv(dest, index=False)
            ok.append(c)
            print(f'  [ok]     {c}  {df.shape[0]} x {df.shape[1]}')
        except Exception as e:
            failed.append(c)
            print(f'  [fail]   {c}  {type(e).__name__}: {e}')
    print(f'\n  {len(ok)}/{len(codes)} components in {out_dir}')
    if failed:
        print(f'  failed: {failed}')
    return out_dir


# --------------------------------------------------------------------------
# locate + load
# --------------------------------------------------------------------------
def _find(data_dir, code):
    """Locate a file whose name contains the NHANES code, ignoring upload prefixes.

    Recursive, because Colab's Drive mount and Kaggle's dataset mount both nest
    files a level or two below the directory you configure.
    """
    hits = sorted(glob.glob(os.path.join(data_dir, '**', f'*{code}*.csv'),
                            recursive=True))
    hits = [h for h in hits if not re.search(r'__\d+_?\.csv$', h)]
    if not hits:
        return None
    return sorted(hits, key=lambda h: (h.count(os.sep), len(h)))[0]


def _count_components(d):
    if not os.path.isdir(d):
        return 0
    return sum(_find(d, c) is not None for c in NHANES_CODES)


def resolve_data_dir(data_dir):
    """Return a directory that actually contains NHANES CSVs, or explain why not."""
    if _count_components(data_dir):
        return data_dir
    for cand in CANDIDATE_DIRS:
        n = _count_components(cand)
        if n:
            print(f'  [auto] no NHANES CSVs in {data_dir!r}; '
                  f'using {cand!r} instead ({n} components found)')
            return cand

    if os.path.isdir(data_dir):
        found = sorted(os.listdir(data_dir))[:25]
        listing = ('\n  that directory contains: '
                   + (', '.join(found) if found else '(nothing)'))
    else:
        listing = '\n  that directory does not exist.'
    raise FileNotFoundError(
        f"No NHANES CSV files found.\n"
        f"  Looked in: {data_dir!r}{listing}\n"
        f"  Also checked: {', '.join(CANDIDATE_DIRS)}\n\n"
        f"  Fix (either one):\n"
        f"    download_nhanes('/content/nhanes')          # fetch from CDC\n"
        f"    CONFIG['data_dir'] = '/content'             # point at your uploads")


def load_nhanes(data_dir):
    """Merge every available NHANES component on SEQN."""
    data_dir = resolve_data_dir(data_dir)
    merged, found = None, []
    for code in NHANES_CODES:
        path = _find(data_dir, code)
        if path is None:
            print(f'  [missing] {code}')
            continue
        df = pd.read_csv(path, low_memory=False)
        df = df.loc[:, ~df.columns.duplicated()]
        # survey weights are design artifacts, never features
        df = df[[c for c in df.columns if not c.startswith(('WT', 'WTS'))]]
        found.append(code)
        merged = df if merged is None else merged.merge(df, on='SEQN', how='outer')

    missing_required = [c for c in REQUIRED if c not in found]
    if missing_required:
        raise FileNotFoundError(
            f"Found {found}, but these are required and absent: {missing_required}.\n"
            f"  GHB_L supplies HbA1c, which IS the target; DEMO_L supplies age/sex.\n"
            f"  Fix: download_nhanes(codes={missing_required})")

    print(f'  loaded {len(found)}/{len(NHANES_CODES)} components -> '
          f'{merged.shape[0]} rows x {merged.shape[1]} cols')
    return merged, found


# --------------------------------------------------------------------------
# clean
# --------------------------------------------------------------------------
def clean_sentinels(df):
    """Fix XPT underflow zeros and NHANES refused/don't-know codes."""
    df = df.copy()
    num = df.select_dtypes(include=[np.number]).columns
    df[num] = df[num].mask(np.isclose(df[num], XPT_ZERO, rtol=1e-3), 0.0)
    for c in df.columns:
        if c.startswith(QUESTIONNAIRE_PREFIX):
            df[c] = df[c].where(~df[c].isin(MISSING_CODES), np.nan)
    return df


def drop_si_duplicates(df, corr_thresh=0.999):
    """NHANES ships many analytes twice: LBX* conventional and LBD*SI in SI units.

    Exact linear rescalings of one another, so keeping both doubles the feature
    count for zero information and splits SHAP attribution across two identical
    columns. Detected empirically, not by name pattern.
    """
    num = df.select_dtypes(include=[np.number]).columns.drop('SEQN', errors='ignore')
    drop, seen = set(), list(num)
    for i, a in enumerate(seen):
        if a in drop:
            continue
        for b in seen[i + 1:]:
            if b in drop:
                continue
            pair = df[[a, b]].dropna()
            if len(pair) < 50:
                continue
            r = abs(np.corrcoef(pair[a], pair[b])[0, 1])
            if r >= corr_thresh:
                drop.add(b if b.endswith('SI') or len(b) > len(a) else a)
    return df.drop(columns=list(drop)), sorted(drop)


# --------------------------------------------------------------------------
# target
# --------------------------------------------------------------------------
def build_target(df, min_age=18):
    """ADA glycemic status from HbA1c: <5.7 normal, 5.7-6.4 prediabetes, >=6.5 diabetes."""
    df = df[df.RIDAGEYR >= min_age].copy()
    df = df.dropna(subset=['LBXGH'])
    df['target'] = pd.cut(df.LBXGH, [-np.inf, 5.7, 6.5, np.inf],
                          right=False, labels=[0, 1, 2]).astype(int)
    return df


def validate_target(df):
    """Cross-check the HbA1c label against self-reported diagnosis (DIQ010).

    DIQ010 is EXCLUDED from the features as leakage, but it is exactly the right
    thing to validate the label against: agreement gives a defensible paragraph
    in the paper, and the disagreement cells are clinically meaningful
    (undiagnosed diabetes vs. well-controlled treated diabetes).
    """
    if 'DIQ010' not in df.columns:
        return None
    sub = df[['target', 'DIQ010']].dropna()
    sub = sub[sub.DIQ010.isin([1, 2, 3])]
    if sub.empty:
        return None
    tab = pd.crosstab(sub.target, sub.DIQ010.map({1: 'yes', 2: 'no', 3: 'borderline'}),
                      rownames=['HbA1c label'], colnames=['self-reported diabetes'])
    lab_diab = sub.target == 2
    self_diab = sub.DIQ010 == 1
    stats_out = {
        'n': len(sub),
        'both_positive': int((lab_diab & self_diab).sum()),
        'lab_only_undiagnosed': int((lab_diab & ~self_diab).sum()),
        'self_only_controlled': int((~lab_diab & self_diab).sum()),
        'agreement': float((lab_diab == self_diab).mean()),
    }
    return tab, stats_out


# --------------------------------------------------------------------------
# leakage control and feature arms
# --------------------------------------------------------------------------
# Always removed: these define the label or are deterministic proxies for it.
LEAKAGE_HARD = [
    'LBXGH', 'LBDGHSI',                    # HbA1c - the label itself
    'LBXGLU', 'LBDGLUSI',                  # fasting plasma glucose
    'LBXSGL', 'LBDSGLSI',                  # serum glucose, same draw
    'LBXGLT', 'LBDGLTSI',                  # OGTT 2-hour glucose
    'SEQN', 'SDDSRVYR', 'RIDSTATR',
    'SDMVSTRA', 'SDMVPSU',                 # survey design
    'RIDAGEMN', 'RIDEXAGM', 'RIDEXMON', 'RIDEXPRG',
]

# The ENTIRE diabetes questionnaire is leakage. DIQ010 is a self-reported
# diagnosis, DIQ050/DIQ070 are insulin and oral-medication use, and DIQ280 is
# the participant's own last A1C reading. Any one of them trivially reveals the
# label. The file is still loaded, because DIQ010 validates the target.
LEAKAGE_PREFIX = ('DIQ', 'DID')

# Defensible either way. Each is a separate switch so the ablation is explicit.
BORDERLINE = {
    # osmolality ~= 2*Na + glucose/18 + BUN/2.8, so glucose is a term inside it
    'osmolality':   ['LBXSOSSI', 'LBXSOSM'],
    # fasting insulin comes from the same fasting draw as glucose
    'insulin':      ['LBXIN', 'LBDINSI', 'LBDINLC'],
    # self-reported clinical management: predicts partly via "this person sees a doctor"
    'care_status':  None,   # resolved at runtime as any questionnaire column
}


def _questionnaire_cols(cols):
    return [c for c in cols if c.startswith(QUESTIONNAIRE_PREFIX)]


FEATURE_ARMS = {
    'full':       dict(drop_race=False, drop_questionnaire=False, drop_borderline=[]),
    'no_race':    dict(drop_race=True,  drop_questionnaire=False, drop_borderline=[]),
    'labs_only':  dict(drop_race=False, drop_questionnaire=True,  drop_borderline=[]),
    'strict':     dict(drop_race=True,  drop_questionnaire=True,
                       drop_borderline=['osmolality', 'insulin']),
}


def make_xy(df, arm='full', max_missing=0.40, add_missing_indicators=True,
            verbose=True):
    """Build the feature matrix for one sensitivity arm.

    add_missing_indicators matters here: several questionnaire items are
    skip-pattern gated (BPQ101D is only asked of people who reported high
    cholesterol), so their missingness is informative. Median-imputing them
    without a flag silently destroys that signal.
    """
    spec = FEATURE_ARMS[arm]
    y = df['target'].values
    X = df.drop(columns=['target'])

    X = X.drop(columns=[c for c in LEAKAGE_HARD if c in X.columns])
    X = X.drop(columns=[c for c in X.columns if c.startswith(LEAKAGE_PREFIX)])
    X = X.select_dtypes(include=[np.number])

    if spec['drop_race']:
        X = X.drop(columns=[c for c in X.columns if c.startswith('RIDRETH')])
    if spec['drop_questionnaire']:
        X = X.drop(columns=_questionnaire_cols(X.columns))
    for key in spec['drop_borderline']:
        cols = BORDERLINE[key] or _questionnaire_cols(X.columns)
        X = X.drop(columns=[c for c in cols if c in X.columns])

    miss = X.isna().mean()
    dropped_missing = miss[miss > max_missing].index.tolist()
    X = X.drop(columns=dropped_missing)

    if add_missing_indicators:
        informative = [c for c in X.columns
                       if 0.02 < X[c].isna().mean() <= max_missing
                       and c.startswith(QUESTIONNAIRE_PREFIX)]
        for c in informative:
            X[f'{c}__isna'] = X[c].isna().astype(int)
    else:
        informative = []

    nunique = X.nunique()
    X = X.drop(columns=nunique[nunique <= 1].index.tolist())

    if verbose:
        print(f'  arm={arm}: dropped {len(dropped_missing)} cols >{max_missing:.0%} '
              f'missing, added {len(informative)} missingness indicators')
        print(f'  feature matrix: {X.shape}')
    return X, y


# ========================================================================
# SECTION 2 - TEN TABULAR MODELS
# ========================================================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.tree import DecisionTreeRegressor

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MIN_EPOCHS = 40      # attention models sit on the majority-class plateau for a long
                     # time before minority classes separate; stopping early on
                     # val loss alone silently yields a majority-class predictor.
MAX_CLASS_WEIGHT = 5.0   # uncapped inverse-frequency weights destabilise SAINT


def class_weights(y, n_classes, cap=MAX_CLASS_WEIGHT):
    counts = np.bincount(y, minlength=n_classes).astype(float)
    counts[counts == 0] = 1.0
    w = len(y) / (n_classes * counts)
    return torch.tensor(np.clip(w, 1.0 / cap, cap), dtype=torch.float32, device=DEVICE)


# --------------------------------------------------------------------------
# shared torch training loop
# --------------------------------------------------------------------------
class TorchClassifier(BaseEstimator, ClassifierMixin):
    """Base class handling training, early stopping, and batching."""

    def __init__(self, n_classes=3, epochs=200, batch_size=256, lr=1e-3,
                 weight_decay=1e-4, patience=25, verbose=False, seed=0,
                 fixed_batch=False):
        self.n_classes = n_classes
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.weight_decay = weight_decay
        self.patience = patience
        self.verbose = verbose
        self.seed = seed
        # SAINT's intersample attention mixes information ACROSS rows in a batch,
        # so predictions depend on batch composition. Such models must see the
        # same batch size at inference, with short batches padded.
        self.fixed_batch = fixed_batch

    def _build(self, n_features):
        raise NotImplementedError

    def fit(self, X, y, X_val=None, y_val=None):
        torch.manual_seed(self.seed)
        np.random.seed(self.seed)
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.int64)
        self.n_features_ = X.shape[1]
        self.model_ = self._build(self.n_features_).to(DEVICE)

        opt = torch.optim.AdamW(self.model_.parameters(), lr=self.lr,
                                weight_decay=self.weight_decay)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.epochs)
        lossf = nn.CrossEntropyLoss(weight=class_weights(y, self.n_classes))

        Xt = torch.tensor(X, device=DEVICE)
        yt = torch.tensor(y, device=DEVICE)
        if X_val is not None:
            Xv = torch.tensor(np.asarray(X_val, dtype=np.float32), device=DEVICE)
            yv = torch.tensor(np.asarray(y_val, dtype=np.int64), device=DEVICE)

        best, best_state, bad = np.inf, None, 0
        n = len(Xt)
        for ep in range(self.epochs):
            self.model_.train()
            perm = torch.randperm(n, device=DEVICE)
            for i in range(0, n, self.batch_size):
                idx = perm[i:i + self.batch_size]
                if self.fixed_batch and len(idx) < self.batch_size:
                    continue                      # drop ragged tail
                opt.zero_grad()
                loss = lossf(self.model_(Xt[idx]), yt[idx])
                loss.backward()
                nn.utils.clip_grad_norm_(self.model_.parameters(), 5.0)
                opt.step()
            sched.step()

            if X_val is not None:
                self.model_.eval()
                with torch.no_grad():
                    vloss = lossf(self._forward_eval(Xv), yv).item()
                if vloss < best - 1e-5:
                    best, bad = vloss, 0
                    best_state = {k: v.detach().clone()
                                  for k, v in self.model_.state_dict().items()}
                else:
                    bad += 1
                    # never stop before MIN_EPOCHS
                    if bad >= self.patience and ep >= MIN_EPOCHS:
                        break
        if best_state is not None:
            self.model_.load_state_dict(best_state)
        return self

    def _forward_eval(self, Xt):
        """Batched forward, padding short batches for batch-dependent models."""
        outs = []
        bs = self.batch_size
        for i in range(0, len(Xt), bs):
            chunk = Xt[i:i + bs]
            if self.fixed_batch and len(chunk) < bs:
                pad = bs - len(chunk)
                chunk = torch.cat([chunk, chunk[-1:].repeat(pad, 1)], dim=0)
                outs.append(self.model_(chunk)[:bs - pad])
            else:
                outs.append(self.model_(chunk))
        return torch.cat(outs, dim=0)

    def predict_proba(self, X):
        self.model_.eval()
        Xt = torch.tensor(np.asarray(X, dtype=np.float32), device=DEVICE)
        with torch.no_grad():
            return F.softmax(self._forward_eval(Xt), dim=1).cpu().numpy()

    def predict(self, X):
        return self.predict_proba(X).argmax(1)


# --------------------------------------------------------------------------
# 1. RealMLP  (Holzmuller et al. 2024)
# --------------------------------------------------------------------------
class RealMLP(TorchClassifier):
    """MLP with scaling-layer input, SELU blocks, and label smoothing."""

    def __init__(self, hidden=256, depth=3, dropout=0.15, **kw):
        super().__init__(**kw)
        self.hidden, self.depth, self.dropout = hidden, depth, dropout

    def _build(self, d):
        layers, prev = [], d
        for _ in range(self.depth):
            layers += [nn.Linear(prev, self.hidden), nn.SELU(),
                       nn.AlphaDropout(self.dropout)]
            prev = self.hidden
        layers.append(nn.Linear(prev, self.n_classes))
        return nn.Sequential(*layers)


# --------------------------------------------------------------------------
# 2. Regularization Cocktail  (Kadra et al. 2021)
# --------------------------------------------------------------------------
class RegCocktail(TorchClassifier):
    """MLP + the cocktail: BN, dropout, weight decay, skip connections, mixup."""

    def __init__(self, hidden=256, depth=4, dropout=0.25, mixup_alpha=0.4, **kw):
        super().__init__(**kw)
        self.hidden, self.depth, self.dropout = hidden, depth, dropout
        self.mixup_alpha = mixup_alpha

    class _Block(nn.Module):
        def __init__(self, d, p):
            super().__init__()
            self.fc = nn.Linear(d, d)
            self.bn = nn.BatchNorm1d(d)
            self.do = nn.Dropout(p)

        def forward(self, x):
            return x + self.do(F.relu(self.bn(self.fc(x))))   # skip connection

    def _build(self, d):
        blocks = [nn.Linear(d, self.hidden), nn.BatchNorm1d(self.hidden), nn.ReLU()]
        blocks += [self._Block(self.hidden, self.dropout) for _ in range(self.depth)]
        blocks.append(nn.Linear(self.hidden, self.n_classes))
        return nn.Sequential(*blocks)


# --------------------------------------------------------------------------
# 3. NAM - Neural Additive Model  (Agarwal et al. 2021)
# --------------------------------------------------------------------------
class NAM(TorchClassifier):
    """One small subnet per feature; output is their sum -> natively interpretable."""

    def __init__(self, unit_hidden=48, **kw):
        super().__init__(**kw)
        self.unit_hidden = unit_hidden

    class _Net(nn.Module):
        """d independent subnets, implemented as grouped 1x1 convolutions.

        A ModuleList of d separate MLPs is mathematically identical but runs d
        sequential kernels per batch; grouped conv does all d in one call, which
        is ~50x faster once d is in the dozens.
        """

        def __init__(self, d, h, n_classes):
            super().__init__()
            self.d, self.n_classes = d, n_classes
            h2 = h // 2
            self.c1 = nn.Conv1d(d, d * h, 1, groups=d)
            self.c2 = nn.Conv1d(d * h, d * h2, 1, groups=d)
            self.c3 = nn.Conv1d(d * h2, d * n_classes, 1, groups=d)
            self.bias = nn.Parameter(torch.zeros(n_classes))

        def _per_feature(self, x):
            z = x.unsqueeze(-1)                       # (B, d, 1)
            z = F.relu(self.c1(z))
            z = F.relu(self.c2(z))
            z = self.c3(z)                            # (B, d*n_classes, 1)
            return z.view(x.size(0), self.d, self.n_classes)

        def forward(self, x):
            return self._per_feature(x).sum(1) + self.bias

        def shape_functions(self, x):
            """Per-feature contribution — the whole point of NAM."""
            return self._per_feature(x)

    def _build(self, d):
        return self._Net(d, self.unit_hidden, self.n_classes)


# --------------------------------------------------------------------------
# 4. DCNv2 - Deep & Cross Network v2  (Wang et al. 2021)
# --------------------------------------------------------------------------
class DCNv2(TorchClassifier):
    def __init__(self, hidden=192, n_cross=3, depth=2, dropout=0.1, **kw):
        super().__init__(**kw)
        self.hidden, self.n_cross, self.depth, self.dropout = hidden, n_cross, depth, dropout

    class _Net(nn.Module):
        def __init__(self, d, hidden, n_cross, depth, p, n_classes):
            super().__init__()
            self.cross_w = nn.ModuleList([nn.Linear(d, d) for _ in range(n_cross)])
            deep, prev = [], d
            for _ in range(depth):
                deep += [nn.Linear(prev, hidden), nn.ReLU(), nn.Dropout(p)]
                prev = hidden
            self.deep = nn.Sequential(*deep)
            self.head = nn.Linear(d + hidden, n_classes)

        def forward(self, x):
            x0, xc = x, x
            for w in self.cross_w:
                xc = x0 * w(xc) + xc          # v2 low-rank cross
            return self.head(torch.cat([xc, self.deep(x)], dim=1))

    def _build(self, d):
        return self._Net(d, self.hidden, self.n_cross, self.depth,
                         self.dropout, self.n_classes)


# --------------------------------------------------------------------------
# 5. DeepFM  (Guo et al. 2017), adapted to continuous fields
# --------------------------------------------------------------------------
class DeepFM(TorchClassifier):
    def __init__(self, emb_dim=12, hidden=192, depth=2, dropout=0.15, **kw):
        super().__init__(**kw)
        self.emb_dim, self.hidden, self.depth, self.dropout = emb_dim, hidden, depth, dropout

    class _Net(nn.Module):
        def __init__(self, d, k, hidden, depth, p, n_classes):
            super().__init__()
            # each continuous field gets its own embedding vector, scaled by value
            self.V = nn.Parameter(torch.randn(d, k) * 0.05)
            self.linear = nn.Linear(d, n_classes)
            deep, prev = [], d * k
            for _ in range(depth):
                deep += [nn.Linear(prev, hidden), nn.ReLU(), nn.Dropout(p)]
                prev = hidden
            self.deep = nn.Sequential(*deep)
            self.head = nn.Linear(hidden, n_classes)
            self.fm_out = nn.Linear(k, n_classes)

        def forward(self, x):
            e = x.unsqueeze(-1) * self.V.unsqueeze(0)        # (B, d, k)
            # FM second order: 0.5 * (sum^2 - sum_of_squares)
            s = e.sum(1)
            fm = 0.5 * (s ** 2 - (e ** 2).sum(1))
            return self.linear(x) + self.fm_out(fm) + self.head(self.deep(e.flatten(1)))

    def _build(self, d):
        return self._Net(d, self.emb_dim, self.hidden, self.depth,
                         self.dropout, self.n_classes)


# --------------------------------------------------------------------------
# 6. TabNet  (Arik & Pfister 2021) - sequential attentive feature selection
# --------------------------------------------------------------------------
def sparsemax(z, dim=-1):
    srt, _ = torch.sort(z, dim=dim, descending=True)
    cum = srt.cumsum(dim) - 1
    k = torch.arange(1, z.size(dim) + 1, device=z.device,
                     dtype=z.dtype).view([-1 if i == dim % z.dim() else 1
                                          for i in range(z.dim())])
    support = (srt - cum / k) > 0
    k_sup = support.sum(dim, keepdim=True).clamp(min=1)
    tau = cum.gather(dim, k_sup - 1) / k_sup.to(z.dtype)
    return torch.clamp(z - tau, min=0)


class TabNet(TorchClassifier):
    def __init__(self, n_d=32, n_steps=3, gamma=1.3, **kw):
        super().__init__(**kw)
        self.n_d, self.n_steps, self.gamma = n_d, n_steps, gamma

    class _Net(nn.Module):
        def __init__(self, d, n_d, n_steps, gamma, n_classes):
            super().__init__()
            self.n_steps, self.gamma, self.n_d = n_steps, gamma, n_d
            self.bn = nn.BatchNorm1d(d)
            self.transformers = nn.ModuleList([
                nn.Sequential(nn.Linear(d, 2 * n_d), nn.BatchNorm1d(2 * n_d), nn.GLU())
                for _ in range(n_steps)])
            self.attentives = nn.ModuleList([
                nn.Sequential(nn.Linear(n_d, d), nn.BatchNorm1d(d))
                for _ in range(n_steps)])
            self.head = nn.Linear(n_d, n_classes)

        def forward(self, x, return_masks=False):
            x = self.bn(x)
            prior = torch.ones_like(x)
            out = torch.zeros(x.size(0), self.n_d, device=x.device)
            a = self.transformers[0](x)
            masks = []
            for step in range(self.n_steps):
                m = sparsemax(self.attentives[step](a) * prior)
                masks.append(m)
                prior = prior * (self.gamma - m)
                a = self.transformers[step](x * m)
                out = out + F.relu(a)
            if return_masks:
                return self.head(out), masks
            return self.head(out)

    def _build(self, d):
        return self._Net(d, self.n_d, self.n_steps, self.gamma, self.n_classes)


# --------------------------------------------------------------------------
# 7. SAINT  (Somepalli et al. 2021) - self + intersample attention
# --------------------------------------------------------------------------
class SAINT(TorchClassifier):
    def __init__(self, emb_dim=24, heads=4, depth=2, dropout=0.1, **kw):
        # intersample attention makes the forward pass batch-dependent
        kw.setdefault('fixed_batch', True)
        super().__init__(**kw)
        self.emb_dim, self.heads, self.depth, self.dropout = emb_dim, heads, depth, dropout

    class _Net(nn.Module):
        def __init__(self, d, k, heads, depth, p, n_classes, proj=96):
            super().__init__()
            self.d, self.k, self.depth = d, k, depth
            self.embed = nn.Parameter(torch.randn(d, k) * 0.05)
            self.bias = nn.Parameter(torch.zeros(d, k))
            self.self_attn = nn.ModuleList(
                [nn.MultiheadAttention(k, heads, dropout=p, batch_first=True)
                 for _ in range(depth)])
            # Intersample attention on the raw d*k flattened row is O((d*k)^2) and
            # blows up for wide tables. Project each row down to `proj` dims first.
            self.down = nn.ModuleList([nn.Linear(d * k, proj) for _ in range(depth)])
            self.inter_attn = nn.ModuleList(
                [nn.MultiheadAttention(proj, heads, dropout=p, batch_first=True)
                 for _ in range(depth)])
            self.up = nn.ModuleList([nn.Linear(proj, d * k) for _ in range(depth)])
            self.ln1 = nn.ModuleList([nn.LayerNorm(k) for _ in range(depth)])
            self.ln2 = nn.ModuleList([nn.LayerNorm(d * k) for _ in range(depth)])
            self.head = nn.Sequential(nn.Linear(d * k, 128), nn.ReLU(),
                                      nn.Linear(128, n_classes))

        def forward(self, x):
            B = x.size(0)
            h = x.unsqueeze(-1) * self.embed.unsqueeze(0) + self.bias.unsqueeze(0)
            for i in range(self.depth):
                # (a) attention across FEATURES within each row
                a, _ = self.self_attn[i](h, h, h)
                h = self.ln1[i](h + a)
                # (b) attention across ROWS within the batch
                flat = h.reshape(B, -1)
                z = self.down[i](flat).unsqueeze(0)          # (1, B, proj)
                b, _ = self.inter_attn[i](z, z, z)
                flat = self.ln2[i](flat + self.up[i](b.squeeze(0)))
                h = flat.reshape(B, self.d, self.k)
            return self.head(h.reshape(B, -1))

    def _build(self, d):
        return self._Net(d, self.emb_dim, self.heads, self.depth,
                         self.dropout, self.n_classes)


# --------------------------------------------------------------------------
# 8. GrowNet  (Badirli et al. 2020) - gradient boosting with weak MLP learners
# --------------------------------------------------------------------------
class GrowNet(BaseEstimator, ClassifierMixin):
    def __init__(self, n_classes=3, n_stages=12, hidden=64, epochs_per_stage=25,
                 lr=1e-2, boost_rate=0.5, seed=0, batch_size=256, **kw):
        self.n_classes, self.n_stages, self.hidden = n_classes, n_stages, hidden
        self.epochs_per_stage, self.lr = epochs_per_stage, lr
        self.boost_rate, self.seed, self.batch_size = boost_rate, seed, batch_size

    def fit(self, X, y, X_val=None, y_val=None):
        torch.manual_seed(self.seed)
        X = torch.tensor(np.asarray(X, np.float32), device=DEVICE)
        y = torch.tensor(np.asarray(y, np.int64), device=DEVICE)
        d = X.shape[1]
        self.stages_, prev_feat = [], None
        ensemble = torch.zeros(len(X), self.n_classes, device=DEVICE)
        w = class_weights(np.asarray(y.cpu()), self.n_classes)

        for s in range(self.n_stages):
            in_dim = d if prev_feat is None else d + self.hidden
            net = nn.Sequential(nn.Linear(in_dim, self.hidden), nn.ReLU()).to(DEVICE)
            head = nn.Linear(self.hidden, self.n_classes).to(DEVICE)
            opt = torch.optim.Adam(list(net.parameters()) + list(head.parameters()),
                                   lr=self.lr)
            # weak learner fits the negative gradient of the current ensemble
            with torch.no_grad():
                resid = F.one_hot(y, self.n_classes).float() - F.softmax(ensemble, 1)
            for _ in range(self.epochs_per_stage):
                perm = torch.randperm(len(X), device=DEVICE)
                for i in range(0, len(X), self.batch_size):
                    idx = perm[i:i + self.batch_size]
                    inp = X[idx] if prev_feat is None else torch.cat(
                        [X[idx], prev_feat[idx]], 1)
                    opt.zero_grad()
                    loss = F.mse_loss(head(net(inp)), resid[idx])
                    loss.backward()
                    opt.step()
            with torch.no_grad():
                inp = X if prev_feat is None else torch.cat([X, prev_feat], 1)
                feat = net(inp)
                ensemble = ensemble + self.boost_rate * head(feat)
                prev_feat = feat
            self.stages_.append((net, head))
        return self

    def predict_proba(self, X):
        X = torch.tensor(np.asarray(X, np.float32), device=DEVICE)
        ens = torch.zeros(len(X), self.n_classes, device=DEVICE)
        prev = None
        with torch.no_grad():
            for net, head in self.stages_:
                inp = X if prev is None else torch.cat([X, prev], 1)
                feat = net(inp)
                ens = ens + self.boost_rate * head(feat)
                prev = feat
            return F.softmax(ens, 1).cpu().numpy()

    def predict(self, X):
        return self.predict_proba(X).argmax(1)


# --------------------------------------------------------------------------
# 9. DeepGBM  (Ke et al. 2019) - GBDT leaf embeddings distilled into an NN
# --------------------------------------------------------------------------
class DeepGBM(BaseEstimator, ClassifierMixin):
    def __init__(self, n_classes=3, n_trees=60, max_depth=4, emb_dim=16,
                 hidden=192, epochs=120, lr=1e-3, batch_size=256, seed=0, **kw):
        self.n_classes, self.n_trees, self.max_depth = n_classes, n_trees, max_depth
        self.emb_dim, self.hidden, self.epochs = emb_dim, hidden, epochs
        self.lr, self.batch_size, self.seed = lr, batch_size, seed

    def _leaves(self, X):
        return np.stack([t.apply(X) for t in self.trees_], 1)

    def fit(self, X, y, X_val=None, y_val=None):
        rng = np.random.RandomState(self.seed)
        torch.manual_seed(self.seed)
        X = np.asarray(X, np.float32)
        y = np.asarray(y, np.int64)
        Y1 = np.eye(self.n_classes)[y]

        # stage 1: grow a small forest on the residuals
        self.trees_, resid = [], Y1 - Y1.mean(0)
        for _ in range(self.n_trees):
            t = DecisionTreeRegressor(max_depth=self.max_depth,
                                      random_state=rng.randint(1 << 30))
            t.fit(X, resid)
            resid = resid - 0.3 * t.predict(X)
            self.trees_.append(t)

        leaves = self._leaves(X)
        self.leaf_offsets_ = np.cumsum([0] + [t.tree_.node_count for t in self.trees_[:-1]])
        idx = leaves + self.leaf_offsets_
        vocab = int(idx.max()) + 1

        # stage 2: embed leaf indices, concatenate with raw features
        class _Net(nn.Module):
            def __init__(self, vocab, n_trees, k, d, hidden, n_classes):
                super().__init__()
                self.emb = nn.Embedding(vocab, k)
                self.net = nn.Sequential(
                    nn.Linear(n_trees * k + d, hidden), nn.ReLU(), nn.Dropout(0.1),
                    nn.Linear(hidden, hidden // 2), nn.ReLU(),
                    nn.Linear(hidden // 2, n_classes))

            def forward(self, leaf_idx, raw):
                e = self.emb(leaf_idx).flatten(1)
                return self.net(torch.cat([e, raw], 1))

        self.model_ = _Net(vocab, self.n_trees, self.emb_dim, X.shape[1],
                           self.hidden, self.n_classes).to(DEVICE)
        opt = torch.optim.AdamW(self.model_.parameters(), lr=self.lr, weight_decay=1e-4)
        lossf = nn.CrossEntropyLoss(weight=class_weights(y, self.n_classes))

        It = torch.tensor(idx, dtype=torch.long, device=DEVICE)
        Xt = torch.tensor(X, device=DEVICE)
        yt = torch.tensor(y, device=DEVICE)
        for _ in range(self.epochs):
            self.model_.train()
            perm = torch.randperm(len(Xt), device=DEVICE)
            for i in range(0, len(Xt), self.batch_size):
                b = perm[i:i + self.batch_size]
                opt.zero_grad()
                lossf(self.model_(It[b], Xt[b]), yt[b]).backward()
                opt.step()
        return self

    def predict_proba(self, X):
        X = np.asarray(X, np.float32)
        idx = self._leaves(X) + self.leaf_offsets_
        self.model_.eval()
        with torch.no_grad():
            out = self.model_(torch.tensor(idx, dtype=torch.long, device=DEVICE),
                              torch.tensor(X, device=DEVICE))
            return F.softmax(out, 1).cpu().numpy()

    def predict(self, X):
        return self.predict_proba(X).argmax(1)


# --------------------------------------------------------------------------
# 10. LightGBM  (reference GBDT baseline)
# --------------------------------------------------------------------------
class LightGBM(BaseEstimator, ClassifierMixin):
    def __init__(self, n_classes=3, n_estimators=500, learning_rate=0.05,
                 num_leaves=31, seed=0, **kw):
        self.n_classes, self.n_estimators = n_classes, n_estimators
        self.learning_rate, self.num_leaves, self.seed = learning_rate, num_leaves, seed

    def fit(self, X, y, X_val=None, y_val=None):
        import lightgbm as lgb
        cw = np.clip(len(y) / (self.n_classes * np.bincount(y, minlength=self.n_classes)),
                     1 / MAX_CLASS_WEIGHT, MAX_CLASS_WEIGHT)
        self.model_ = lgb.LGBMClassifier(
            objective='multiclass', num_class=self.n_classes,
            n_estimators=self.n_estimators, learning_rate=self.learning_rate,
            num_leaves=self.num_leaves, random_state=self.seed, verbose=-1,
            class_weight={i: cw[i] for i in range(self.n_classes)})
        if X_val is not None:
            import lightgbm as lgb2
            self.model_.fit(X, y, eval_set=[(X_val, y_val)],
                            callbacks=[lgb2.early_stopping(50, verbose=False)])
        else:
            self.model_.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

    def predict(self, X):
        return self.model_.predict(X)


MODEL_ZOO = {
    'LightGBM':    LightGBM,
    'DeepGBM':     DeepGBM,
    'GrowNet':     GrowNet,
    'TabNet':      TabNet,
    'SAINT':       SAINT,
    'NAM':         NAM,
    'DCNv2':       DCNv2,
    'DeepFM':      DeepFM,
    'RealMLP':     RealMLP,
    'RegCocktail': RegCocktail,
}


# ========================================================================
# SECTION 3 - METRICS, EXPLAINABILITY, STATISTICS
# ========================================================================

import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             precision_score, recall_score, matthews_corrcoef,
                             cohen_kappa_score, roc_auc_score, average_precision_score,
                             log_loss, confusion_matrix, brier_score_loss)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')


# ==========================================================================
# 1. METRICS
# ==========================================================================
def all_metrics(y_true, y_pred, y_prob, n_classes=3):
    """Every metric that is defensible for imbalanced multi-class tabular work."""
    Yb = label_binarize(y_true, classes=list(range(n_classes)))
    m = {
        'accuracy':            accuracy_score(y_true, y_pred),
        'balanced_accuracy':   balanced_accuracy_score(y_true, y_pred),
        'f1_macro':            f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_micro':            f1_score(y_true, y_pred, average='micro', zero_division=0),
        'f1_weighted':         f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'precision_macro':     precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro':        recall_score(y_true, y_pred, average='macro', zero_division=0),
        'mcc':                 matthews_corrcoef(y_true, y_pred),
        'cohen_kappa':         cohen_kappa_score(y_true, y_pred),
        'kappa_quadratic':     cohen_kappa_score(y_true, y_pred, weights='quadratic'),
        'log_loss':            log_loss(y_true, y_prob, labels=list(range(n_classes))),
    }
    # AUC family — one-vs-rest, guarded against single-class folds
    try:
        m['roc_auc_ovr_macro'] = roc_auc_score(y_true, y_prob, multi_class='ovr',
                                               average='macro')
        m['roc_auc_ovo_macro'] = roc_auc_score(y_true, y_prob, multi_class='ovo',
                                               average='macro')
    except ValueError:
        m['roc_auc_ovr_macro'] = m['roc_auc_ovo_macro'] = np.nan
    try:
        m['pr_auc_macro'] = average_precision_score(Yb, y_prob, average='macro')
    except ValueError:
        m['pr_auc_macro'] = np.nan

    # multi-class Brier score (calibration)
    m['brier'] = float(np.mean(np.sum((y_prob - Yb) ** 2, axis=1)))

    # per-class detail
    per_f1 = f1_score(y_true, y_pred, average=None, labels=list(range(n_classes)),
                      zero_division=0)
    per_rec = recall_score(y_true, y_pred, average=None, labels=list(range(n_classes)),
                           zero_division=0)
    per_prec = precision_score(y_true, y_pred, average=None, labels=list(range(n_classes)),
                               zero_division=0)
    for c in range(n_classes):
        m[f'f1_class{c}'] = per_f1[c]
        m[f'recall_class{c}'] = per_rec[c]
        m[f'precision_class{c}'] = per_prec[c]

    # specificity per class, from the confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    for c in range(n_classes):
        tn = cm.sum() - cm[c, :].sum() - cm[:, c].sum() + cm[c, c]
        fp = cm[:, c].sum() - cm[c, c]
        m[f'specificity_class{c}'] = tn / (tn + fp) if (tn + fp) else np.nan
    return m


# ==========================================================================
# 2. EXPLAINABILITY
# ==========================================================================
def shap_values(model, X_bg, X_explain, feature_names, model_name,
                n_bg=50, nsamples=100):
    """SHAP values with the fastest correct explainer for each model family.

    TreeExplainer is exact and fast for LightGBM. Everything else needs the
    model-agnostic KernelExplainer, which is O(nsamples * n_bg) forward passes,
    so both are kept small and the explanation set is subsampled.
    """
    import shap
    try:
        if model_name == 'LightGBM':
            expl = shap.TreeExplainer(model.model_)
            sv = expl.shap_values(X_explain)
            if isinstance(sv, list):
                sv = np.stack(sv, axis=-1)
            elif sv.ndim == 2:
                sv = sv[:, :, None]
            return sv                                   # (n, d, n_classes)

        bg = shap.kmeans(X_bg, min(n_bg, len(X_bg)))
        expl = shap.KernelExplainer(model.predict_proba, bg)
        sv = expl.shap_values(X_explain, nsamples=nsamples, silent=True)
        if isinstance(sv, list):
            sv = np.stack(sv, axis=-1)
        return sv
    except Exception as e:
        print(f'    [shap] {model_name}: {type(e).__name__}: {e}')
        return None


def shap_importance(sv, feature_names):
    """Mean |SHAP| per feature, averaged over classes."""
    if sv is None:
        return None
    imp = np.abs(sv).mean(axis=(0, 2)) if sv.ndim == 3 else np.abs(sv).mean(0)
    return pd.Series(imp, index=feature_names).sort_values(ascending=False)


def lime_explanations(model, X_train, X_explain, feature_names, class_names,
                      n_instances=5, n_features=10, seed=0):
    """Local surrogate explanations for a handful of individual patients."""
    from lime.lime_tabular import LimeTabularExplainer
    try:
        expl = LimeTabularExplainer(
            X_train, feature_names=list(feature_names), class_names=list(class_names),
            discretize_continuous=True, mode='classification', random_state=seed)
        out = []
        for i in range(min(n_instances, len(X_explain))):
            e = expl.explain_instance(X_explain[i], model.predict_proba,
                                      num_features=n_features,
                                      labels=list(range(len(class_names))))
            row = {'instance': i}
            for lbl in range(len(class_names)):
                row[f'class{lbl}'] = dict(e.as_list(label=lbl))
            out.append(row)
        return out
    except Exception as e:
        print(f'    [lime] {type(e).__name__}: {e}')
        return None


def lime_importance(lime_out, feature_names):
    """Aggregate LIME weights into a global-ish ranking (mean |weight|)."""
    if not lime_out:
        return None
    acc = {}
    for row in lime_out:
        for k, d in row.items():
            if k == 'instance':
                continue
            for feat_rule, w in d.items():
                # LIME returns discretized rules like "LBXSUA > 5.20"; map back
                base = next((f for f in feature_names if f in feat_rule), feat_rule)
                acc.setdefault(base, []).append(abs(w))
    return pd.Series({k: np.mean(v) for k, v in acc.items()}).sort_values(ascending=False)


def nam_shape_functions(model, X, feature_names):
    """NAM's built-in interpretability: exact per-feature contribution curves."""
    import torch
    try:
        model.model_.eval()
        Xt = torch.tensor(np.asarray(X, np.float32), device=next(model.model_.parameters()).device)
        with torch.no_grad():
            sf = model.model_.shape_functions(Xt).cpu().numpy()   # (n, d, n_classes)
        return sf
    except Exception as e:
        print(f'    [nam] {type(e).__name__}: {e}')
        return None


def tabnet_masks(model, X):
    """TabNet's sparse attention masks = native feature selection per step."""
    import torch
    try:
        model.model_.eval()
        Xt = torch.tensor(np.asarray(X, np.float32), device=next(model.model_.parameters()).device)
        with torch.no_grad():
            _, masks = model.model_(Xt, return_masks=True)
        return np.stack([m.cpu().numpy() for m in masks], 0).mean(0).mean(0)
    except Exception as e:
        print(f'    [tabnet] {type(e).__name__}: {e}')
        return None


# ==========================================================================
# 3. STATISTICAL COMPARISON
# ==========================================================================
def descriptive_stats(df, metric):
    """Mean, SD, and 95% CI of a metric per model across folds."""
    g = df.groupby('model')[metric]
    out = g.agg(['count', 'mean', 'std', 'min', 'max'])
    se = out['std'] / np.sqrt(out['count'])
    tcrit = stats.t.ppf(0.975, out['count'] - 1)
    out['ci95_low'] = out['mean'] - tcrit * se
    out['ci95_high'] = out['mean'] + tcrit * se
    return out.sort_values('mean', ascending=False)


def assumption_checks(df, metric):
    """ANOVA assumes normal residuals and equal variances. Test both."""
    groups = [g[metric].values for _, g in df.groupby('model')]
    rows = []
    for name, g in df.groupby('model'):
        v = g[metric].dropna().values
        if len(v) >= 3:
            W, p = stats.shapiro(v)
            rows.append({'model': name, 'shapiro_W': W, 'shapiro_p': p,
                         'normal_at_0.05': p > 0.05})
    lev_stat, lev_p = stats.levene(*groups)
    return pd.DataFrame(rows), {'levene_stat': lev_stat, 'levene_p': lev_p,
                                'equal_variance_at_0.05': lev_p > 0.05}


def anova_tukey(df, metric):
    """One-way ANOVA across models + Tukey HSD post-hoc."""
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    groups = [g[metric].values for _, g in df.groupby('model')]
    F, p = stats.f_oneway(*groups)

    # effect size: eta-squared
    grand = df[metric].mean()
    ss_between = sum(len(g) * (g.mean() - grand) ** 2 for g in groups)
    ss_total = ((df[metric] - grand) ** 2).sum()
    eta2 = ss_between / ss_total if ss_total else np.nan

    tukey = pairwise_tukeyhsd(df[metric].values, df['model'].values, alpha=0.05)
    tk = pd.DataFrame(tukey._results_table.data[1:],
                      columns=tukey._results_table.data[0])
    return {'F': F, 'p': p, 'eta_squared': eta2}, tk


def friedman_nemenyi(df, metric):
    """Non-parametric alternative — the standard test for comparing classifiers.

    Friedman ranks models within each fold, so it does not assume normality or
    equal variance, and it respects the fact that folds are paired across models.
    """
    wide = df.pivot_table(index='fold_id', columns='model', values=metric)
    wide = wide.dropna()
    if wide.shape[0] < 3 or wide.shape[1] < 3:
        return None, None, None
    stat, p = stats.friedmanchisquare(*[wide[c].values for c in wide.columns])
    ranks = wide.rank(axis=1, ascending=False).mean().sort_values()

    # Nemenyi critical difference
    k, n = wide.shape[1], wide.shape[0]
    q_alpha = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949,
               8: 3.031, 9: 3.102, 10: 3.164, 11: 3.219, 12: 3.268}.get(k, 3.164)
    cd = q_alpha * np.sqrt(k * (k + 1) / (6 * n))
    return {'friedman_stat': stat, 'friedman_p': p, 'critical_difference': cd}, ranks, wide


def feature_target_tests(X, y, feature_names):
    """Univariate screening: ANOVA F per feature + Kruskal-Wallis, FDR-corrected."""
    from statsmodels.stats.multitest import multipletests
    rows = []
    for j, f in enumerate(feature_names):
        col = X[:, j]
        groups = [col[y == c] for c in np.unique(y)]
        groups = [g[~np.isnan(g)] for g in groups]
        if min(len(g) for g in groups) < 3:
            continue
        F, pF = stats.f_oneway(*groups)
        H, pH = stats.kruskal(*groups)
        # eta-squared as effect size
        grand = np.concatenate(groups).mean()
        ssb = sum(len(g) * (g.mean() - grand) ** 2 for g in groups)
        sst = ((np.concatenate(groups) - grand) ** 2).sum()
        rows.append({'feature': f, 'anova_F': F, 'anova_p': pF,
                     'kruskal_H': H, 'kruskal_p': pH,
                     'eta_squared': ssb / sst if sst else np.nan})
    out = pd.DataFrame(rows)
    if len(out):
        out['anova_p_fdr'] = multipletests(out['anova_p'], method='fdr_bh')[1]
        out['significant_fdr'] = out['anova_p_fdr'] < 0.05
        out = out.sort_values('anova_F', ascending=False)
    return out


# ==========================================================================
# SECTION 4 — CROSS-VALIDATION RUNNER
# ==========================================================================
import os, time, json
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import QuantileTransformer

CLASS_NAMES = ['Normal', 'Prediabetes', 'Diabetes']


def build_preprocessor(X_train):
    """Fit imputer + scaler on TRAINING data only.

    Fitting either on the full dataset leaks test-fold distribution information
    into training and inflates every score. This must live inside the CV loop.
    """
    imp = SimpleImputer(strategy='median').fit(X_train)
    sc = QuantileTransformer(output_distribution='normal',
                             n_quantiles=min(1000, len(X_train)),
                             random_state=0).fit(imp.transform(X_train))
    return imp, sc


def apply_preprocessor(imp, sc, X):
    return sc.transform(imp.transform(X))


def make_model(name, cfg, seed):
    cls = MODEL_ZOO[name]
    kw = dict(n_classes=cfg['n_classes'], seed=seed)
    if isinstance(cls, type) and issubclass(cls, TorchClassifier):
        kw.update(epochs=cfg['epochs'], patience=cfg['patience'],
                  batch_size=cfg['batch_size'])
    elif name == 'GrowNet':
        kw.update(n_stages=cfg['grownet_stages'],
                  epochs_per_stage=cfg['grownet_epochs'])
    elif name == 'DeepGBM':
        kw.update(epochs=cfg['epochs'])
    return cls(**kw)


def run_cv(X_df, y, cfg, arm='full'):
    """Repeated stratified K-fold across every model in the zoo."""
    X = X_df.values
    feature_names = list(X_df.columns)
    rskf = RepeatedStratifiedKFold(n_splits=cfg['n_folds'],
                                   n_repeats=cfg['n_repeats'],
                                   random_state=cfg['seed'])
    records, oof_store, fitted_last = [], {}, {}
    total = cfg['n_folds'] * cfg['n_repeats']

    for fold_i, (tr_idx, te_idx) in enumerate(rskf.split(X, y)):
        repeat, fold = divmod(fold_i, cfg['n_folds'])
        imp, sc = build_preprocessor(X[tr_idx])
        Xtr = apply_preprocessor(imp, sc, X[tr_idx])
        Xte = apply_preprocessor(imp, sc, X[te_idx])
        ytr, yte = y[tr_idx], y[te_idx]

        # inner split for early stopping — never touches the test fold
        Xtr2, Xva, ytr2, yva = train_test_split(
            Xtr, ytr, test_size=0.15, stratify=ytr, random_state=cfg['seed'])

        print(f'\n[{arm}] fold {fold_i + 1}/{total}  repeat {repeat} fold {fold}  '
              f'train={len(Xtr2)} val={len(Xva)} test={len(Xte)}')

        for name in cfg['models']:
            t0 = time.time()
            try:
                m = make_model(name, cfg, seed=cfg['seed'] + fold_i)
                m.fit(Xtr2, ytr2, Xva, yva)
                prob = m.predict_proba(Xte)
                pred = prob.argmax(1)
                met = all_metrics(yte, pred, prob, cfg['n_classes'])
                met.update(model=name, arm=arm, repeat=repeat, fold=fold,
                           fold_id=f'r{repeat}f{fold}', fit_seconds=time.time() - t0)
                records.append(met)
                oof_store.setdefault(name, []).append(
                    {'test_idx': te_idx, 'y_true': yte, 'y_pred': pred, 'y_prob': prob})
                if fold_i == total - 1:
                    fitted_last[name] = (m, Xtr2, ytr2, Xte, yte, feature_names)
                print(f'   {name:12s} F1m={met["f1_macro"]:.4f} '
                      f'balAcc={met["balanced_accuracy"]:.4f} '
                      f'AUC={met["roc_auc_ovr_macro"]:.4f} '
                      f'({met["fit_seconds"]:.0f}s)')
            except Exception as e:
                print(f'   {name:12s} FAILED: {type(e).__name__}: {e}')

    return pd.DataFrame(records), oof_store, fitted_last


# ==========================================================================
# SECTION 5 — EXPLAINABILITY ORCHESTRATION
# ==========================================================================
def run_xai(fitted_last, cfg, outdir):
    """SHAP + LIME for every model, plus each model's native interpretability."""
    shap_table, lime_table, native = {}, {}, {}

    for name, (m, Xtr, ytr, Xte, yte, feat) in fitted_last.items():
        print(f'\n  explaining {name} ...')
        n_exp = cfg['shap_n_explain'] if name == 'LightGBM' else cfg['shap_n_explain_nn']

        sv = shap_values(m, Xtr, Xte[:n_exp], feat, name,
                         n_bg=cfg['shap_n_background'], nsamples=cfg['shap_nsamples'])
        imp = shap_importance(sv, feat)
        if imp is not None:
            shap_table[name] = imp
            np.save(os.path.join(outdir, f'shap_values_{name}.npy'), sv)
            print(f'    SHAP top-5: {", ".join(imp.head(5).index)}')

        lo = lime_explanations(m, Xtr, Xte, feat, CLASS_NAMES,
                               n_instances=cfg['lime_n_instances'],
                               n_features=cfg['lime_n_features'], seed=cfg['seed'])
        li = lime_importance(lo, feat)
        if li is not None:
            lime_table[name] = li
            with open(os.path.join(outdir, f'lime_local_{name}.json'), 'w') as f:
                json.dump([{k: (v if k == 'instance' else
                               {kk: float(vv) for kk, vv in v.items()})
                            for k, v in r.items()} for r in lo], f, indent=2)
            print(f'    LIME top-5: {", ".join(li.head(5).index)}')

        if name == 'NAM':
            sf = nam_shape_functions(m, Xte, feat)
            if sf is not None:
                native['NAM_shape_functions'] = sf
                np.save(os.path.join(outdir, 'nam_shape_functions.npy'), sf)
                native['NAM_importance'] = pd.Series(
                    np.abs(sf).mean(axis=(0, 2)), index=feat).sort_values(ascending=False)
        if name == 'TabNet':
            mk = tabnet_masks(m, Xte)
            if mk is not None:
                native['TabNet_masks'] = pd.Series(mk, index=feat).sort_values(ascending=False)
        if name == 'LightGBM':
            native['LightGBM_gain'] = pd.Series(
                m.model_.booster_.feature_importance('gain'),
                index=feat).sort_values(ascending=False)

    shap_df = pd.DataFrame(shap_table).fillna(0) if shap_table else pd.DataFrame()
    lime_df = pd.DataFrame(lime_table).fillna(0) if lime_table else pd.DataFrame()
    return shap_df, lime_df, native


def xai_agreement(shap_df, lime_df, top_k=15):
    """Do SHAP and LIME agree? Rank correlation + top-k overlap per model."""
    rows = []
    for m in shap_df.columns:
        if m not in lime_df.columns:
            continue
        common = shap_df.index.intersection(lime_df.index)
        s, l = shap_df.loc[common, m], lime_df.loc[common, m]
        rho, p = stats.spearmanr(s, l)
        overlap = len(set(s.nlargest(top_k).index) & set(l.nlargest(top_k).index))
        rows.append({'model': m, 'spearman_rho': rho, 'p_value': p,
                     f'top{top_k}_overlap': overlap,
                     f'top{top_k}_jaccard': overlap / (2 * top_k - overlap)})
    return pd.DataFrame(rows).sort_values('spearman_rho', ascending=False)


# ==========================================================================
# SECTION 6 — FIGURES
# ==========================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

sns.set_theme(style='whitegrid', context='paper')


def fig_metric_comparison(res, metric, outdir):
    order = res.groupby('model')[metric].mean().sort_values(ascending=False).index
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=res, x='model', y=metric, order=order, errorbar='sd',
                capsize=.15, ax=ax, hue='model', legend=False, palette='crest')
    sns.stripplot(data=res, x='model', y=metric, order=order, ax=ax,
                  color='0.25', size=3, alpha=.55, jitter=.22)
    ax.set_title(f'{metric} across {res.fold_id.nunique()} folds (mean ± SD)')
    ax.set_xlabel(''); plt.xticks(rotation=35, ha='right'); plt.tight_layout()
    fig.savefig(os.path.join(outdir, f'fig_{metric}.png'), dpi=200)
    plt.close(fig)


def fig_confusions(oof_store, outdir, n_classes=3):
    names = list(oof_store)
    if not names:
        return
    ncol = min(5, len(names))
    nrow = int(np.ceil(len(names) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 3.1 * nrow),
                             squeeze=False)
    for ax, name in zip(np.ravel(axes), names):
        yt = np.concatenate([f['y_true'] for f in oof_store[name]])
        yp = np.concatenate([f['y_pred'] for f in oof_store[name]])
        cm = confusion_matrix(yt, yp, labels=range(n_classes), normalize='true')
        ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
            ax=ax, cmap='Blues', colorbar=False, values_format='.2f')
        ax.set_title(name, fontsize=10)
        ax.tick_params(labelsize=7)
    for ax in np.ravel(axes)[len(names):]:
        ax.axis('off')
    plt.tight_layout()
    fig.savefig(os.path.join(outdir, 'fig_confusion_matrices.png'), dpi=200)
    plt.close(fig)


def fig_shap_heatmap(shap_df, outdir, top_k=20, fname='fig_shap_heatmap.png'):
    if shap_df.empty:
        return
    norm = shap_df / shap_df.sum()
    top = norm.mean(1).nlargest(top_k).index
    fig, ax = plt.subplots(figsize=(1.05 * shap_df.shape[1] + 4, .34 * top_k + 2))
    sns.heatmap(norm.loc[top], cmap='rocket_r', ax=ax,
                cbar_kws={'label': 'normalised mean |SHAP|'})
    ax.set_title(f'Top-{top_k} features, all models')
    plt.tight_layout()
    fig.savefig(os.path.join(outdir, fname), dpi=200)
    plt.close(fig)


def fig_nam_shapes(native, feature_names, X_ref, outdir, top_k=9):
    sf = native.get('NAM_shape_functions')
    if sf is None:
        return
    imp = native['NAM_importance'].head(top_k)
    fig, axes = plt.subplots(3, 3, figsize=(12, 9))
    for ax, feat in zip(np.ravel(axes), imp.index):
        j = feature_names.index(feat)
        col = X_ref[:len(sf), j]
        order = np.argsort(col)
        for c in range(sf.shape[2]):
            ax.plot(col[order], sf[order, j, c], label=CLASS_NAMES[c], lw=1.4)
        ax.set_title(feat, fontsize=9)
        ax.axhline(0, color='0.6', lw=.7, ls='--')
        ax.tick_params(labelsize=7)
    np.ravel(axes)[0].legend(fontsize=7)
    fig.suptitle('NAM learned shape functions (exact, not post-hoc)', y=1.0)
    plt.tight_layout()
    fig.savefig(os.path.join(outdir, 'fig_nam_shape_functions.png'), dpi=200)
    plt.close(fig)


def fig_critical_difference(ranks, cd, outdir):
    if ranks is None:
        return
    fig, ax = plt.subplots(figsize=(9, 2.6))
    ax.scatter(ranks.values, np.zeros(len(ranks)), s=55, zorder=3, color='#2b6cb0')
    for i, (name, r) in enumerate(ranks.items()):
        ax.annotate(name, (r, 0), xytext=(0, 14 if i % 2 == 0 else -20),
                    textcoords='offset points', ha='center', fontsize=8)
    best = ranks.min()
    ax.plot([best, best + cd], [0.045, 0.045], lw=3, color='crimson')
    ax.annotate(f'CD = {cd:.2f}', ((2 * best + cd) / 2, 0.06), ha='center',
                fontsize=9, color='crimson')
    ax.set_yticks([]); ax.set_xlabel('mean rank (lower is better)')
    ax.set_title('Nemenyi critical-difference diagram')
    ax.grid(axis='x', alpha=.3); plt.tight_layout()
    fig.savefig(os.path.join(outdir, 'fig_critical_difference.png'), dpi=200)
    plt.close(fig)


def fig_class_balance(y, outdir):
    fig, ax = plt.subplots(figsize=(5, 3.4))
    counts = pd.Series(y).value_counts().sort_index()
    labels = [CLASS_NAMES[i] for i in counts.index]
    sns.barplot(x=labels, y=counts.values, ax=ax, hue=labels, legend=False,
                palette='crest')
    for i, v in enumerate(counts.values):
        ax.text(i, v, f'{v}\n({v / counts.sum():.1%})', ha='center', va='bottom',
                fontsize=9)
    ax.set_ylabel('n participants'); ax.set_title('Target distribution')
    ax.margins(y=.18); plt.tight_layout()
    fig.savefig(os.path.join(outdir, 'fig_class_balance.png'), dpi=200)
    plt.close(fig)


def fig_arm_comparison(all_res, metric, outdir):
    """Side-by-side model performance across sensitivity arms."""
    if all_res.arm.nunique() < 2:
        return
    order = (all_res[all_res.arm == 'full'].groupby('model')[metric]
             .mean().sort_values(ascending=False).index)
    fig, ax = plt.subplots(figsize=(11, 5))
    sns.barplot(data=all_res, x='model', y=metric, hue='arm', order=order,
                errorbar='sd', capsize=.08, ax=ax, palette='Set2')
    ax.set_title(f'{metric} by feature arm — leakage / fairness sensitivity analysis')
    ax.set_xlabel(''); plt.xticks(rotation=35, ha='right')
    ax.legend(title='arm', fontsize=8); plt.tight_layout()
    fig.savefig(os.path.join(outdir, f'fig_arm_comparison_{metric}.png'), dpi=200)
    plt.close(fig)


# ==========================================================================
# SECTION 7 — MAIN
# ==========================================================================
CONFIG = {
    # data ---------------------------------------------------------------
    'data_dir':            './nhanes_csv',
    'download':            True,    # fetch missing components from CDC first
    'outdir':              './results',
    # task ---------------------------------------------------------------
    'n_classes':           3,
    'min_age':             18,
    'arms':                ['full', 'labs_only', 'strict'],
    'xai_arm':             'full',  # run SHAP/LIME on this arm only (it is slow)
    'add_missing_indicators': True,
    # cv -----------------------------------------------------------------
    'n_folds':             5,
    'n_repeats':           5,       # 5x5 = 25 folds. Set to 1 for a quick pass.
    'seed':                42,
    # models -------------------------------------------------------------
    'epochs':              200,
    'patience':            25,
    'batch_size':          256,
    'grownet_stages':      12,
    'grownet_epochs':      25,
    # xai ----------------------------------------------------------------
    'shap_n_background':   50,
    'shap_n_explain':      500,     # tree models are cheap
    'shap_n_explain_nn':   60,      # KernelSHAP is not
    'shap_nsamples':       120,
    'lime_n_instances':    10,
    'lime_n_features':     12,
    # reporting ----------------------------------------------------------
    'primary_metric':      'f1_macro',
    'models': ['LightGBM', 'DeepGBM', 'GrowNet', 'TabNet', 'SAINT',
               'NAM', 'DCNv2', 'DeepFM', 'RealMLP', 'RegCocktail'],
}


def run_statistics(res, cfg, outdir, tag=''):
    """Assumption checks, ANOVA + Tukey, Friedman + Nemenyi."""
    metric = cfg['primary_metric']
    summary = descriptive_stats(res, metric)
    print(summary.round(4).to_string())

    sh, lev = assumption_checks(res, metric)
    print('\n-- Shapiro-Wilk normality per model --')
    print(sh.round(4).to_string(index=False))
    print(f'-- Levene equal-variance: W={lev["levene_stat"]:.4f} '
          f'p={lev["levene_p"]:.4f} -> equal_variance={lev["equal_variance_at_0.05"]}')

    aov, tukey = anova_tukey(res, metric)
    print(f'\n-- One-way ANOVA: F={aov["F"]:.4f}  p={aov["p"]:.3e}  '
          f'eta2={aov["eta_squared"]:.4f}')
    print(tukey.to_string(index=False))
    tukey.to_csv(os.path.join(outdir, f'tukey_hsd{tag}.csv'), index=False)

    fr, ranks, _ = friedman_nemenyi(res, metric)
    if fr:
        print(f'\n-- Friedman: chi2={fr["friedman_stat"]:.4f} '
              f'p={fr["friedman_p"]:.3e}  CD={fr["critical_difference"]:.3f}')
        print(ranks.round(3).to_string())
        fig_critical_difference(ranks, fr['critical_difference'], outdir)
        ranks.to_frame('mean_rank').to_csv(
            os.path.join(outdir, f'friedman_ranks{tag}.csv'))
    return summary


def main(cfg=CONFIG):
    os.makedirs(cfg['outdir'], exist_ok=True)
    t_start = time.time()

    print('=' * 74)
    print('STEP 1  ACQUIRE AND MERGE NHANES COMPONENTS')
    print('=' * 74)
    if cfg.get('download'):
        download_nhanes(cfg['data_dir'])
    df, found = load_nhanes(cfg['data_dir'])
    df = clean_sentinels(df)
    df, dropped_si = drop_si_duplicates(df)
    print(f'  dropped {len(dropped_si)} collinear SI-unit duplicates')

    print('\n' + '=' * 74)
    print('STEP 2  BUILD AND VALIDATE TARGET')
    print('=' * 74)
    df = build_target(df, min_age=cfg['min_age'])
    print(f'  adults {cfg["min_age"]}+ with HbA1c: {len(df)}')
    y_all = df['target'].values
    print('  class distribution:',
          {CLASS_NAMES[i]: int(n) for i, n in enumerate(np.bincount(y_all))})
    fig_class_balance(y_all, cfg['outdir'])

    vt = validate_target(df)
    if vt is not None:
        tab, vstats = vt
        print('\n  HbA1c label vs self-reported diagnosis (DIQ010):')
        print(tab.to_string())
        print(f'  agreement={vstats["agreement"]:.3f}  '
              f'undiagnosed (lab+/self-)={vstats["lab_only_undiagnosed"]}  '
              f'treated-controlled (lab-/self+)={vstats["self_only_controlled"]}')
        tab.to_csv(os.path.join(cfg['outdir'], 'target_validation.csv'))
    else:
        print('  [skip] DIQ_L not available — no self-report cross-check')

    all_res, arm_summaries = [], {}
    for arm in cfg['arms']:
        arm_dir = os.path.join(cfg['outdir'], arm)
        os.makedirs(arm_dir, exist_ok=True)

        print('\n' + '=' * 74)
        print(f'ARM "{arm}"  —  FEATURE MATRIX')
        print('=' * 74)
        X_df, y = make_xy(df, arm=arm,
                          add_missing_indicators=cfg['add_missing_indicators'])
        X_df.assign(target=y).to_csv(
            os.path.join(arm_dir, 'analysis_dataset.csv'), index=False)

        print('\n  -- univariate screening (ANOVA F + Kruskal-Wallis, BH-FDR) --')
        imp0, sc0 = build_preprocessor(X_df.values)
        uni = feature_target_tests(apply_preprocessor(imp0, sc0, X_df.values),
                                   y, list(X_df.columns))
        uni.to_csv(os.path.join(arm_dir, 'univariate_tests.csv'), index=False)
        print(uni.head(10).round(4).to_string(index=False))
        print(f'  {int(uni.significant_fdr.sum())}/{len(uni)} features significant '
              f'after Benjamini-Hochberg correction')

        print('\n' + '=' * 74)
        print(f'ARM "{arm}"  —  {cfg["n_repeats"]}x{cfg["n_folds"]} REPEATED '
              f'STRATIFIED CV, {len(cfg["models"])} MODELS')
        print('=' * 74)
        res, oof, fitted = run_cv(X_df, y, cfg, arm=arm)
        res.to_csv(os.path.join(arm_dir, 'cv_results_raw.csv'), index=False)
        all_res.append(res)

        print('\n' + '-' * 74)
        print(f'ARM "{arm}"  —  RESULTS AND STATISTICS')
        print('-' * 74)
        arm_summaries[arm] = run_statistics(res, cfg, arm_dir)

        metric_cols = [c for c in res.columns
                       if c not in ('model', 'arm', 'repeat', 'fold', 'fold_id')]
        res.groupby('model')[metric_cols].agg(['mean', 'std']).to_csv(
            os.path.join(arm_dir, 'cv_results_summary.csv'))
        for mt in ['f1_macro', 'balanced_accuracy', 'roc_auc_ovr_macro', 'mcc']:
            if mt in res.columns:
                fig_metric_comparison(res, mt, arm_dir)
        fig_confusions(oof, arm_dir, cfg['n_classes'])

        if arm == cfg['xai_arm']:
            print('\n' + '-' * 74)
            print(f'ARM "{arm}"  —  EXPLAINABILITY: SHAP + LIME + NATIVE')
            print('-' * 74)
            shap_df, lime_df, native = run_xai(fitted, cfg, arm_dir)
            if not shap_df.empty:
                shap_df.to_csv(os.path.join(arm_dir, 'shap_importance.csv'))
                fig_shap_heatmap(shap_df, arm_dir)
                print('\n-- consensus SHAP ranking across models --')
                print((shap_df / shap_df.sum()).mean(1).nlargest(15).round(4).to_string())
            if not lime_df.empty:
                lime_df.to_csv(os.path.join(arm_dir, 'lime_importance.csv'))
                fig_shap_heatmap(lime_df, arm_dir, fname='fig_lime_heatmap.png')
            if not shap_df.empty and not lime_df.empty:
                agree = xai_agreement(shap_df, lime_df)
                print('\n-- SHAP vs LIME agreement --')
                print(agree.round(4).to_string(index=False))
                agree.to_csv(os.path.join(arm_dir, 'xai_agreement.csv'), index=False)
            for k, v in native.items():
                if isinstance(v, pd.Series):
                    v.to_csv(os.path.join(arm_dir, f'native_{k}.csv'))
            if 'NAM' in fitted:
                fig_nam_shapes(native, fitted['NAM'][5], fitted['NAM'][3], arm_dir)

    print('\n' + '=' * 74)
    print('CROSS-ARM SENSITIVITY ANALYSIS')
    print('=' * 74)
    combined = pd.concat(all_res, ignore_index=True)
    combined.to_csv(os.path.join(cfg['outdir'], 'cv_results_all_arms.csv'), index=False)
    pivot = combined.pivot_table(index='model', columns='arm',
                                 values=cfg['primary_metric'], aggfunc='mean')
    if 'full' in pivot.columns:
        for a in pivot.columns:
            if a != 'full':
                pivot[f'delta_{a}'] = pivot[a] - pivot['full']
    print(pivot.round(4).to_string())
    pivot.to_csv(os.path.join(cfg['outdir'], 'arm_comparison.csv'))
    for mt in ['f1_macro', 'balanced_accuracy']:
        fig_arm_comparison(combined, mt, cfg['outdir'])

    with open(os.path.join(cfg['outdir'], 'config.json'), 'w') as f:
        json.dump({k: v for k, v in cfg.items()}, f, indent=2)

    print('\n' + '=' * 74)
    print(f'DONE in {(time.time() - t_start) / 60:.1f} min  ->  {cfg["outdir"]}/')
    print('=' * 74)
    for root, _, files in os.walk(cfg['outdir']):
        for fn in sorted(files):
            print('   ', os.path.relpath(os.path.join(root, fn), cfg['outdir']))
    return combined, arm_summaries


if __name__ == '__main__':
    results, summaries = main()

STEP 1  ACQUIRE AND MERGE NHANES COMPONENTS
  [ok]     DEMO_L  11933 x 27
  [ok]     BIOPRO_L  7199 x 42
  [ok]     CBC_L  8727 x 23
  [ok]     GHB_L  7199 x 3
  [ok]     GLU_L  3996 x 4
  [ok]     INS_L  3996 x 5
  [ok]     TRIGLY_L  3996 x 10
  [ok]     TCHOL_L  8068 x 4
  [ok]     HDL_L  8068 x 4
  [ok]     HSCRP_L  8727 x 4
  [ok]     VID_L  8727 x 10
  [ok]     FERTIN_L  2564 x 4
  [ok]     TFR_L  2564 x 4
  [ok]     BMX_L  8860 x 22
  [ok]     BPXO_L  7801 x 12
  [ok]     MCQ_L  11744 x 35
  [ok]     BPQ_L  8501 x 6
  [ok]     DIQ_L  11744 x 9

  18/18 components in ./nhanes_csv
  loaded 18/18 components -> 11933 rows x 197 cols
  dropped 20 collinear SI-unit duplicates

STEP 2  BUILD AND VALIDATE TARGET
  adults 18+ with HbA1c: 6002
  class distribution: {'Normal': 3691, 'Prediabetes': 1590, 'Diabetes': 721}

  HbA1c label vs self-reported diagnosis (DIQ010):
self-reported diabetes  borderline    no  yes
HbA1c label                                  
0                            